# BERT から文脈に応じた単語ベクトルを取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_transformers/contextual_word_vectors.ipynb)

NLP クックブック 08-02 ／ Transformers の基本操作

同じ単語でも文脈によって意味が違う（bank = 銀行 / 土手）。文脈を反映した単語ベクトルを取り出して比べたい。

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased").eval()

def word_vector(sentence, word):
    words = sentence.split()
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**enc).last_hidden_state[0]          # （トークン数, 768）
    positions = [i for i, w in enumerate(enc.word_ids()) if w == words.index(word)]
    return hidden[positions].mean(dim=0)                    # サブワードが複数なら平均

sents = ["i deposited money in the bank yesterday", "the bank raised its interest rates",
         "we had a picnic on the grassy bank of the river",
         "the river overflowed its bank after the heavy rain"]
vecs = torch.stack([word_vector(s, "bank") for s in sents])
sim = torch.nn.functional.cosine_similarity(vecs[:, None], vecs[None], dim=-1)
for s, row in zip(sents, sim):
    print(" ".join(f"{x:.2f}" for x in row), "|", s)

1.00 0.71 0.43 0.50 | i deposited money in the bank yesterday
0.71 1.00 0.40 0.49 | the bank raised its interest rates
0.43 0.40 1.00 0.74 | we had a picnic on the grassy bank of the river
0.50 0.49 0.74 1.00 | the river overflowed its bank after the heavy rain


- 金融の bank 同士（文 1 と 2）、川岸の bank 同士（文 3 と 4）の類似度が 0.7 程度と高く、語義が違う組み合わせ（0.4〜0.5）より高くなる。word2vec では 1 つのベクトルに混ざっていた語義が分離される。
- 例文によっては、同じ語義同士の類似度が上がらず、語義が違う組み合わせより低くなることもある。どの層を使うかでも性質が変わる。`output_hidden_states=True` で全層を取り出し、最後の 4 層を平均する方法もよく使われる。